# RTENSO2026: sudden stratospheric warmings and the polar vortex, 2026 starts

The Northern Hemisphere polar vortex in the E3SM RTENSO2026 hindcasts from the 1 Feb, 1 May and
1 Aug 2026 starts, against ERA5. This covers the end of winter 2025/26 (Feb start only) and the
forecast winters 2026/27 (El Niño) and 2027/28.

* **Model climatology** (section 3): the seasonal cycle of the 10-hPa vortex wind and the
  winter zonal-mean wind in the Reanalysis (BruteForce) 1981-2010 hindcasts against ERA5.
  This bias affects how many SSWs the forecasts can produce.
* **QBO** (section 4): equatorial zonal wind from each start against ERA5, and how long
  the initial QBO phase persists.
* **Winter 2025/26** (section 5): the observed vortex and final warming, and the 1 Feb 2026 start.
* **2026/27 and 2027/28 forecasts** (section 6): daily U(60°N, 10 hPa) plumes, SSW counts per
  member-winter and final-warming dates against ERA5 1979-2026, and tercile probabilities of
  the DJF vortex strength.
* **ENSO and QBO across members** (section 7): member DJF vortex against member Niño3.4 and
  autumn QBO, next to the observed winters.
* **SSW composites** (section 8): U(60°N, 10 hPa), the 100-hPa eddy heat flux and the polar-cap
  height ("dripping paint") around the central dates, pooled across all model SSWs, against ERA5 1979-2025.

**Data.** Model daily values are the native `eam.h1` daily means (U/V/T at 22 levels up to 1 hPa,
Z at 16 levels up to 10 hPa), which have not been post-processed. They are averaged into 1° latitude
bands (`util/strat_case.py`) and cached under `$S2D_DIAG_ROOT/<source>/strat/`. Monthly values
come from the post-processed 1x1 U, T, Z3 (19 levels). ERA5 is the daily zonal-mean archive
`ERA5/1x1/daily_zonal` (zonal means of the 00Z and 12Z analyses on 37 levels, built by
`ERA5/scripts/daily_zonal`). It runs to the last complete month on the NERSC mirror, so the 2026/27
winter is not yet observed.

**Definitions.** SSW: the daily zonal-mean U at 60°N (the 59.5/60.5°N bands), 10 hPa, turns
easterly between 1 Nov and 31 Mar. Events must be separated by 20 consecutive westerly days, and the
wind must return to westerly for 10 consecutive days before 30 Apr (Charlton and Polvani 2007; Butler
et al. 2017). Final warming: the spring onset of easterlies after which the wind does not stay
westerly for 10 days before 31 Jul. Model SSWs are counted on the raw model wind
and on a **bias-adjusted** wind (raw minus the difference between the model and ERA5 climatologies
on that day), because a too-strong or too-weak model vortex changes how often it reverses.

**Anomalies.** Model anomalies remove the Reanalysis 1981-2010 lead-time climatology
(May/Nov starts; Feb/Aug use `CLIM_METHOD = "lead_drift"`: the ERA5 target-month climatology plus
the mean May/Nov drift at the same lead). Daily model anomalies use that monthly climatology
interpolated to days, and daily ERA5 anomalies use a smoothed 1981-2010 daily climatology
(4 harmonics). Polar-cap height anomalies are divided by the ERA5 daily standard deviation for both.

**Pooled ensembles:** each start pools the `POOL` cases (30 members). The 1 Feb 2026 `clim` and `obs`
runs are bit-identical (see "1 Feb 2026 start" in `README.md`), so that start pools 20 members.

In [ ]:
%matplotlib inline

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.lines import Line2D

from util import rtenso_case as rc
from util import strat_case as sc

warnings.filterwarnings("ignore", message="invalid value encountered")
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="All-NaN slice")

### Run controls

In [ ]:
NOTEBOOK = "9_ssw_polar_vortex"   # outputs: rc.OUTPUT_ROOT/9_ssw_polar_vortex/{figures,data}/
# "auto": reuse the zonal-mean caches; "rebuild": re-read the model output; "require": fail if a cache is missing
derivation_mode = "auto"
CLIM_METHOD = "lead_drift"        # Feb/Aug drift correction: "lead_drift" or "target_month"
WORKERS = 8                       # processes reading model output (cache builds only)

# Data locations (machine-specific)
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")                       # reference-data archive
ERA5_ZONAL_DIR = OBSERVATION_ROOT / "ERA5" / "1x1" / "daily_zonal"                    # ERA5 daily zonal means
SIMULATION_ROOT = Path("/global/cfs/cdirs/e3sm/S2S2D/RTENSO2026/simulation")         # raw eam.h1 history

YEAR = 2026
INIT_MONTHS = (2, 5, 8)
MONTH_NAMES = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
               9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
INIT_COLORS = {2: "#1f77b4", 5: "#2ca02c", 8: "#d62728"}
MODEL_PREFIX = rc.MODEL_PREFIX
POOL = {
    "Reanalysis-CLIM": f"{MODEL_PREFIX}_RTENSO2026_clim",
    "Reanalysis-OBS": f"{MODEL_PREFIX}_RTENSO2026_obs",
    "Reanalysis-NSIDC": f"{MODEL_PREFIX}_RTENSO2026_obs_ice_nsidc",
}

CLIM_YEARS = rc.CLIM_YEARS          # 1981-2010: model drift and ERA5 anomaly climatology
ERA5_FIRST_YEAR = 1979              # ERA5 daily record read (SSW climatology from winter 1979/80)
REF_LEVEL = 10.0                    # hPa, SSW wind
QBO_LEVEL = 50.0                    # hPa, QBO phase (Holton-Tan)
QBO_MONTHS = (10, 11)               # QBO phase months before the winter
DJF = ((0, 12), (1, 1), (1, 2))     # (years after the winter year, month)
SSW = {"sep_days": 20, "recover_days": 10}
NINO_PRODUCT_OBS = "HadISST1.1"     # observed DJF Nino3.4 (1870 on)
ENSO_THRESHOLD = 0.5                # degC: El Nino / La Nina winters (DJF Nino3.4)
LAGS = np.arange(-40, 61)           # days around SSW central dates
PCH_LAGS = (-30, 60)

fontz = 13
fig_type = "png"
fig_dpi = 200
plt.rcParams.update({"font.size": fontz, "axes.titlesize": fontz, "axes.labelsize": fontz,
                     "legend.fontsize": fontz * 0.8, "xtick.labelsize": fontz * 0.85, "ytick.labelsize": fontz * 0.85})


def save(fig, *parts):
    path = rc.save_figure(fig, NOTEBOOK, *parts, dpi=fig_dpi, ext=fig_type)
    print(f"[FIGURE] {path}")


DATA_DIR = rc.output_dir(NOTEBOOK, "data")

### 1. ERA5

Daily zonal means from `ERA5_FIRST_YEAR` to the last complete month, the smoothed 1981-2010 daily
climatology and standard deviation of each index, and calendar-month means for the QBO and the
monthly drift climatology.

In [ ]:
%%time
era5 = sc.era5_daily_zonal(ERA5_ZONAL_DIR, years=(ERA5_FIRST_YEAR, YEAR))
obs_end = pd.Timestamp(era5["time"].values[-1])
print(f"ERA5 daily zonal means {pd.Timestamp(era5['time'].values[0]):%d %b %Y} - {obs_end:%d %b %Y}")


def doy_of(da):
    return xr.DataArray(sc.noleap_doy(da["time"].values), dims="time", coords={"time": da["time"]})


def on_days(clim, da):
    # a (doy, ...) climatology on the days of da
    return clim.sel(doy=doy_of(da)).drop_vars("doy")


# indices
e_u60 = sc.lat_mean(era5["U"].sel(plev=REF_LEVEL), sc.U60)              # (time)
e_pcz = sc.lat_mean(era5["Z"], sc.PCAP)                                  # (time, plev)
e_pct = sc.lat_mean(era5["T"].sel(plev=REF_LEVEL), sc.PCAP)
e_vt = sc.lat_mean(era5["VT"].sel(plev=100.0), sc.VT_BAND)
e_qbo = sc.monthly_means(sc.lat_mean(era5["U"], sc.EQ))                  # (time, plev), monthly

clim_u60, std_u60 = sc.daily_climatology(e_u60, CLIM_YEARS)
clim_pcz, std_pcz = sc.daily_climatology(e_pcz, CLIM_YEARS)
clim_vt, _ = sc.daily_climatology(e_vt, CLIM_YEARS)
clim_pct, _ = sc.daily_climatology(e_pct, CLIM_YEARS)
e_pch = (e_pcz - on_days(clim_pcz, e_pcz)) / on_days(std_pcz, e_pcz)    # standardized polar-cap height

# calendar-month means and their 1981-2010 climatology (U, T, Z) for the model drift correction
e_month_clim = sc.obs_month_climatology(era5[["U", "T", "Z"]], CLIM_YEARS)
e_u60_month = sc.monthly_means(e_u60)

### 2. Model

* **Monthly zonal means**: the Reanalysis 1981-2010 May/Nov starts (drift climatology and the
  model's own interannual spread) and each 2026 start (pooled members).
* **Daily zonal means** of each 2026 start (pooled members) from the native `eam.h1` files, with the
  monthly lead-time climatology interpolated to days. The first build reads about 90 x 25 history
  files (about 15 min with 8 processes); later runs read the caches.

In [ ]:
%%time
ref = {m: sc.monthly_zonal(rc.REFERENCE["case_prefix"], "Reanalysis", range(CLIM_YEARS[0], CLIM_YEARS[1] + 1), m,
                           mode=derivation_mode, workers=WORKERS)
       for m in rc.REFERENCE["init_months"]}
MPLEV = ref[rc.REFERENCE["init_months"][0]]["plev"].values
e_month_clim_m = e_month_clim.sel(plev=MPLEV)            # ERA5 on the model monthly levels (all are ERA5 levels)
lead_clim = {init: sc.lead_climatology(init, ref, e_month_clim_m, method=CLIM_METHOD) for init in INIT_MONTHS}


def source_of(prefix):
    return prefix.split(f"{MODEL_PREFIX}_")[-1]


def target_time(init, nlead=rc.NLEAD):
    out = []
    for lead in range(nlead):
        m = init - 1 + lead
        out.append(np.datetime64(f"{YEAR + m // 12:04d}-{m % 12 + 1:02d}-15"))
    return np.asarray(out, dtype="datetime64[ns]")


def pooled_monthly(init):
    # pooled monthly zonal means (member, lead, plev, lat) of one start; repeats counted once
    parts = []
    for name, prefix in POOL.items():
        ds = sc.monthly_zonal(prefix, source_of(prefix), [YEAR], init, mode=derivation_mode, workers=WORKERS)
        if ds is None:
            continue
        ds = ds.isel(year=0, drop=True)
        ds = ds.isel(member=np.flatnonzero(np.isfinite(ds["U"].sel(plev=REF_LEVEL).isel(lead=0, lat=-1)).values))
        u = ds["U"].sel(plev=REF_LEVEL).values
        if any(p.sizes["member"] == ds.sizes["member"] and np.array_equal(u, p["U"].sel(plev=REF_LEVEL).values, equal_nan=True)
               for p in parts):
            print(f"[monthly {name} init{init:02d}] repeats an already pooled case; not pooled again")
            continue
        parts.append(ds.assign_coords(member=[f"{name}:{m}" for m in ds["member"].values]))
    out = xr.concat(parts, dim="member")
    return out.assign_coords(time=("lead", target_time(init, out.sizes["lead"])))


monthly = {init: pooled_monthly(init) for init in INIT_MONTHS}
daily = {init: sc.pooled_daily_zonal(SIMULATION_ROOT, POOL, YEAR, init, mode=derivation_mode, workers=WORKERS)
         for init in INIT_MONTHS}
for init in INIT_MONTHS:
    d = daily[init]
    print(f"[init {MONTH_NAMES[init]} {YEAR}] {d.sizes['member']} members, daily "
          f"{pd.Timestamp(d['time'].values[0]):%d %b %Y} - {pd.Timestamp(d['time'].values[-1]):%d %b %Y}; "
          f"monthly {monthly[init].sizes['member']} members; climatology: {lead_clim[init].attrs['climatology_method']}")

In [ ]:
# Daily model indices, the model climatology on the same days, and the ERA5 climatology on those days
m_u60, m_u60_clim, m_u60_adj, m_pch, m_vt = {}, {}, {}, {}, {}
std_pcz_m = sc.interp_logp(std_pcz, sc.DAILY_LEVELS)
for init in INIT_MONTHS:
    d = daily[init]
    cd = sc.daily_from_lead(sc.interp_logp(lead_clim[init][["U", "Z"]], sc.DAILY_LEVELS), YEAR, init, d["time"].values)
    cd = cd.assign_coords(time=d["time"])
    m_u60[init] = sc.lat_mean(d["U"].sel(plev=REF_LEVEL), sc.U60)                      # (member, time)
    m_u60_clim[init] = sc.lat_mean(cd["U"].sel(plev=REF_LEVEL), sc.U60)                # (time)
    # bias-adjusted: remove the model-minus-ERA5 climatological difference of each day
    m_u60_adj[init] = m_u60[init] - (m_u60_clim[init] - on_days(clim_u60, m_u60_clim[init]))
    pcz = sc.lat_mean(d["Z"], sc.PCAP) - sc.lat_mean(cd["Z"], sc.PCAP)                  # (member, time, plev)
    m_pch[init] = (pcz / on_days(std_pcz_m, pcz)).dropna("plev", how="all")
    m_vt[init] = sc.lat_mean(d["VT"].sel(plev=100.0), sc.VT_BAND)

# Monthly model indices: U60 at REF_LEVEL (raw and anomaly) and equatorial U (QBO)
mm_u60, mm_u60_anom, mm_qbo = {}, {}, {}
for init in INIT_MONTHS:
    mo = monthly[init]
    mm_u60[init] = sc.lat_mean(mo["U"].sel(plev=REF_LEVEL), sc.U60)
    mm_u60_anom[init] = mm_u60[init] - sc.lat_mean(lead_clim[init]["U"].sel(plev=REF_LEVEL), sc.U60)
    mm_qbo[init] = sc.lat_mean(mo["U"], sc.EQ)                                          # (member, lead, plev)

### 3. Model stratospheric climatology against ERA5

Monthly U(60°N, 10 hPa) and 10-hPa polar-cap temperature by calendar month: ERA5 1981-2010
(mean and ±1 standard deviation of the monthly means) and the Reanalysis hindcasts at each
lead (May and Nov starts, all years and members, so the shading is the model's total spread). Right:
DJF zonal-mean U, the model at leads 1-3 of the Nov starts minus ERA5 (shading) with ERA5 contours.

In [ ]:
fig = plt.figure(figsize=(17, 5.2))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.15], wspace=0.28)
ref_colors = {5: "tab:orange", 11: "tab:purple"}
for k, (field, name, units) in enumerate([("U", "U(60°N, 10 hPa)", "m s$^{-1}$"), ("T", "T(60-90°N, 10 hPa)", "K")]):
    ax = fig.add_subplot(gs[k])
    band = sc.U60 if field == "U" else sc.PCAP
    e_series = sc.monthly_means(sc.lat_mean(era5[field].sel(plev=REF_LEVEL), band))
    e_series = e_series.sel(time=slice(f"{CLIM_YEARS[0]}-01", f"{CLIM_YEARS[1]}-12"))
    g = e_series.groupby("time.month")
    order = [7, 8, 9, 10, 11, 12, 1, 2, 3, 4, 5, 6]
    x = np.arange(12)
    mean, std = g.mean().sel(month=order).values, g.std().sel(month=order).values
    ax.fill_between(x, mean - std, mean + std, color="0.8", label="ERA5 ±1σ")
    ax.plot(x, mean, "k-", lw=2.5, label=f"ERA5 {CLIM_YEARS[0]}-{CLIM_YEARS[1]}")
    for m, ds in ref.items():
        idx = sc.lat_mean(ds[field].sel(plev=REF_LEVEL), band)                       # (year, member, lead)
        months = sc.target_months(m)
        for half in (0, 1):                                                          # leads 0-11, 12-23
            leads = np.arange(12 * half, 12 * half + 12)
            xs = np.array([order.index(months[l]) for l in leads])
            srt = np.argsort(xs)
            mu = idx.isel(lead=leads).mean(("year", "member")).values[srt]
            sd = idx.isel(lead=leads).std(("year", "member")).values[srt]
            ls = "-" if half == 0 else ":"
            ax.plot(xs[srt], mu, ls, color=ref_colors[m], lw=2,
                    label=f"E3SM {MONTH_NAMES[m]} starts, leads {leads[0]}-{leads[-1]}")
            if half == 0:
                ax.fill_between(xs[srt], mu - sd, mu + sd, color=ref_colors[m], alpha=0.15, lw=0)
    if field == "U":
        ax.axhline(0, color="0.4", lw=0.8)
    ax.set_xticks(x, [MONTH_NAMES[m][0] for m in order])
    ax.set_title(f"({'ab'[k]}) {name}", loc="left")
    ax.set_ylabel(units)
    ax.grid(alpha=0.3)
    if k == 0:
        ax.legend(loc="upper left", fontsize=fontz * 0.65, frameon=False)

ax = fig.add_subplot(gs[2])
nov = ref[11]["U"].isel(lead=[1, 2, 3]).mean(("year", "member", "lead"))
obs_djf = e_month_clim_m["U"].sel(month=[12, 1, 2]).mean("month")
diff = (nov - obs_djf).sel(lat=slice(0, 90))
levels = np.arange(-12, 13, 2)
cf = ax.contourf(diff["lat"], diff["plev"], diff, levels=levels, cmap="RdBu_r", extend="both")
cs = ax.contour(obs_djf["lat"].sel(lat=slice(0, 90)), obs_djf["plev"], obs_djf.sel(lat=slice(0, 90)),
                levels=np.arange(-40, 81, 10), colors="k", linewidths=0.8)
ax.clabel(cs, fmt="%d", fontsize=fontz * 0.6)
ax.set_yscale("log")
ax.set_ylim(1000, 1)
ax.set_yticks([1000, 300, 100, 30, 10, 3, 1], ["1000", "300", "100", "30", "10", "3", "1"])
ax.set_xlabel("Latitude (°N)")
ax.set_ylabel("hPa")
ax.set_title("(c) DJF zonal-mean U: E3SM (Nov starts, leads 1-3) − ERA5", loc="left", fontsize=fontz * 0.9)
fig.colorbar(cf, ax=ax, label="m s$^{-1}$", shrink=0.9)
save(fig, "climatology", "u60_t60_10hPa", "djf_u_bias")
plt.show()

clim_table = pd.DataFrame({
    "ERA5": e_u60_month.sel(time=slice(f"{CLIM_YEARS[0]}-01", f"{CLIM_YEARS[1]}-12")).groupby("time.month").mean().to_series(),
    **{f"E3SM {MONTH_NAMES[m]} starts (leads 0-11)": pd.Series(
        sc.lat_mean(ds["U"].sel(plev=REF_LEVEL), sc.U60).isel(lead=slice(0, 12)).mean(("year", "member")).values,
        index=sc.target_months(m)[:12]) for m, ds in ref.items()},
}).loc[[10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9]].round(1)
clim_table.index = [MONTH_NAMES[m] for m in clim_table.index]
print(f"U(60N, {REF_LEVEL:g} hPa) monthly climatology, m/s")
display(clim_table.T)

### 4. QBO

Monthly zonal-mean U over 5°S-5°N: ERA5 (top), then the ensemble mean of each start with the ERA5
values over the same months as contours. The right panels show the `QBO_LEVEL` wind of every member
against ERA5. A forecast QBO that decays, or that stops descending, moves the Holton-Tan pathway
(easterly lower-stratospheric QBO, weaker vortex) away from the observed one.

In [ ]:
qbo_span = (np.datetime64(f"{YEAR - 1}-01-01"), np.datetime64(f"{YEAR + 2}-08-01"))
fig, axes = plt.subplots(len(INIT_MONTHS) + 1, 2, figsize=(17, 3.0 * (len(INIT_MONTHS) + 1)),
                         gridspec_kw={"width_ratios": [1.6, 1], "hspace": 0.45, "wspace": 0.18}, sharex="col")
qlev = np.arange(-35, 36, 5)
eq = e_qbo.sel(time=slice(*qbo_span), plev=slice(1, 100))
ax = axes[0, 0]
cf = ax.contourf(eq["time"], eq["plev"], eq.T, levels=qlev, cmap="RdBu_r", extend="both")
ax.set_title(f"(a) ERA5 zonal-mean U, 5°S-5°N (through {obs_end:%b %Y})", loc="left")
axes[0, 1].axis("off")
for k, init in enumerate(INIT_MONTHS, start=1):
    q = mm_qbo[init].sel(plev=slice(1, 100))
    qt = q.assign_coords(time=monthly[init]["time"]).swap_dims(lead="time")
    ax = axes[k, 0]
    ax.contourf(qt["time"], qt["plev"], qt.mean("member").T, levels=qlev, cmap="RdBu_r", extend="both")
    ov = eq.sel(time=slice(qt["time"].values[0], qt["time"].values[-1]))
    if ov.sizes["time"] > 1:
        cs = ax.contour(ov["time"], ov["plev"], ov.T, levels=[-20, -10, 0, 10], colors="k", linewidths=0.8)
        ax.clabel(cs, fmt="%d", fontsize=fontz * 0.55)
    ax.axvline(np.datetime64(f"{YEAR}-{init:02d}-01"), color="k", lw=1.5)
    ax.set_title(f"({'abcdefgh'[2 * k - 1]}) E3SM {MONTH_NAMES[init]} {YEAR} start, ensemble mean "
                 f"({monthly[init].sizes['member']} members; contours: ERA5)", loc="left", fontsize=fontz * 0.85)
    ax2 = axes[k, 1]
    q50 = qt.sel(plev=QBO_LEVEL)
    for mem in q50["member"].values:
        ax2.plot(q50["time"], q50.sel(member=mem), color=INIT_COLORS[init], lw=0.6, alpha=0.5)
    ax2.plot(q50["time"], q50.mean("member"), color=INIT_COLORS[init], lw=2.5)
    e50 = e_qbo.sel(plev=QBO_LEVEL, time=slice(*qbo_span))
    ax2.plot(e50["time"], e50, "k-", lw=2)
    ax2.axhline(0, color="0.4", lw=0.8)
    ax2.set_ylim(-35, 25)
    ax2.set_title(f"({'abcdefgh'[2 * k]}) U {QBO_LEVEL:g} hPa, 5°S-5°N", loc="left", fontsize=fontz * 0.85)
    ax2.grid(alpha=0.3)
for ax in axes[:, 0]:
    ax.set_yscale("log")
    ax.set_ylim(100, 1)
    ax.set_yticks([100, 50, 30, 10, 3, 1], ["100", "50", "30", "10", "3", "1"])
    ax.set_ylabel("hPa")
    ax.set_xlim(*qbo_span)
for ax in axes[-1]:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
fig.colorbar(cf, ax=axes[:, 0], label="m s$^{-1}$", shrink=0.6, pad=0.01)
save(fig, "qbo", YEAR)
plt.show()

# QBO phase before each forecast winter: member QBO_LEVEL wind over QBO_MONTHS
qbo_rows = []
for init in INIT_MONTHS:
    qt = mm_qbo[init].sel(plev=QBO_LEVEL).assign_coords(time=monthly[init]["time"]).swap_dims(lead="time")
    for w in (YEAR, YEAR + 1):
        sel = [t for t in pd.DatetimeIndex(qt["time"].values) if t.year == w and t.month in QBO_MONTHS]
        if len(sel) == len(QBO_MONTHS):
            v = qt.sel(time=sel).mean("time")
            qbo_rows.append({"start": MONTH_NAMES[init], "winter": f"{w}/{str(w + 1)[2:]}",
                             f"U{QBO_LEVEL:g} mean (m/s)": float(v.mean()), "members easterly": int((v < 0).sum()),
                             "members": int(v.size)})
e_qbo_on = {w: float(e_qbo.sel(plev=QBO_LEVEL).sel(time=[np.datetime64(f"{w}-{m:02d}-15") for m in QBO_MONTHS]).mean())
            for w in range(ERA5_FIRST_YEAR, YEAR + 1)
            if pd.Timestamp(f"{w}-{max(QBO_MONTHS):02d}-15") <= obs_end}
print(f"QBO {QBO_LEVEL:g} hPa over months {QBO_MONTHS}; ERA5 last observed autumn {max(e_qbo_on)}: {e_qbo_on[max(e_qbo_on)]:+.1f} m/s")
display(pd.DataFrame(qbo_rows).round(1))

### 5. Winter 2025/26 and the 1 Feb 2026 start

Observed daily U(60°N, 10 hPa) from 1 Oct 2025 with the ERA5 1981-2010 daily climatology
(mean ±1σ and ±2σ), the members and ensemble mean of the 1 Feb 2026 start, and the observed and
member final-warming dates (bottom rug). Below: the observed standardized polar-cap height and the Feb
start's ensemble mean over the same days.

In [ ]:
w0 = YEAR - 1
span = (pd.Timestamp(w0, 10, 1), pd.Timestamp(YEAR, 6, 30))
e_obs = e_u60.sel(time=slice(*span))
cl = on_days(clim_u60, e_obs)
sd = on_days(std_u60, e_obs)
obs_ssw_w0 = [d for d in sc.ssw_central_dates(e_u60.values, e_u60["time"].values, **SSW) if sc.winter_of(d) == w0]
obs_fw = {y: sc.final_warming_date(e_u60.values, e_u60["time"].values, y, recover_days=SSW["recover_days"])
          for y in range(ERA5_FIRST_YEAR, obs_end.year + 1)}
fw_text = f"{obs_fw[YEAR]:%d %b}" if pd.notna(obs_fw.get(YEAR, pd.NaT)) else "not yet"
print(f"ERA5 winter {w0}/{str(YEAR)[2:]}: SSW central dates {[f'{d:%d %b %Y}' for d in obs_ssw_w0] or 'none'}; "
      f"final warming {fw_text}")

init = 2
has_feb = init in INIT_MONTHS
fig, axes = plt.subplots(3 if has_feb else 2, 1, figsize=(15, 12 if has_feb else 8.5),
                         gridspec_kw={"height_ratios": [1.3, 1, 1][: 3 if has_feb else 2], "hspace": 0.32}, sharex=True)
ax = axes[0]
t = e_obs["time"].values
ax.fill_between(t, cl - 2 * sd, cl + 2 * sd, color="0.9", label="ERA5 ±2σ")
ax.fill_between(t, cl - sd, cl + sd, color="0.78", label="ERA5 ±1σ")
ax.plot(t, cl, color="0.4", lw=1.5, label=f"ERA5 {CLIM_YEARS[0]}-{CLIM_YEARS[1]} mean")
if has_feb:
    u = m_u60[init].sel(time=slice(*span))
    for mem in u["member"].values:
        ax.plot(u["time"], u.sel(member=mem), color=INIT_COLORS[init], lw=0.6, alpha=0.45)
    ax.plot(u["time"], u.mean("member"), color=INIT_COLORS[init], lw=2.5, label=f"E3SM 1 Feb {YEAR} ({u.sizes['member']} members)")
    ax.plot(u["time"], m_u60_clim[init].sel(time=slice(*span)), "--", color=INIT_COLORS[init], lw=1.5,
            label="E3SM lead-time climatology")
ax.plot(t, e_obs, "k-", lw=2.5, label="ERA5")
for d in obs_ssw_w0:
    ax.axvline(d, color="k", ls=":", lw=1.2)
    ax.text(d, 55, f" SSW {d:%d %b}", fontsize=fontz * 0.7, va="top")
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("m s$^{-1}$")
ax.set_ylim(-30, 60)
ax.set_title(f"(a) U(60°N, {REF_LEVEL:g} hPa), winter {w0}/{str(YEAR)[2:]}", loc="left")
# final-warming rug
if has_feb:
    fw_m = [sc.final_warming_date(m_u60[init].sel(member=mem).values, m_u60[init]["time"].values, YEAR,
                                  recover_days=SSW["recover_days"]) for mem in m_u60[init]["member"].values]
    fw_m = [d for d in fw_m if pd.notna(d)]
    ax.plot(fw_m, np.full(len(fw_m), -27), "|", color=INIT_COLORS[init], ms=14, mew=2, label="final warming (members)")
if pd.notna(obs_fw.get(YEAR, pd.NaT)):
    ax.plot([obs_fw[YEAR]], [-27], "kv", ms=11, label="final warming (ERA5)")
ax.legend(loc="upper right", ncol=2, fontsize=fontz * 0.65, frameon=False)
ax.grid(alpha=0.3)

pch_lev = np.array([-2.5, -2, -1.5, -1, -0.5, 0.5, 1, 1.5, 2, 2.5])
ep = e_pch.sel(time=slice(*span)).sel(plev=slice(1, 1000))
ax = axes[1]
cf = ax.contourf(ep["time"], ep["plev"], ep.T, levels=pch_lev, cmap="RdBu_r", extend="both")
ax.set_title("(b) ERA5 polar-cap (60-90°N) height anomaly / σ", loc="left")
if has_feb:
    mp = m_pch[init].sel(time=slice(*span)).mean("member")
    ax = axes[2]
    ax.contourf(mp["time"], mp["plev"], mp.T, levels=pch_lev, cmap="RdBu_r", extend="both")
    ax.set_title(f"(c) E3SM 1 Feb {YEAR} ensemble mean, same quantity (ERA5 σ)", loc="left")
for ax in axes[1:]:
    ax.set_yscale("log")
    ax.set_ylim(1000, 1)
    ax.set_yticks([1000, 300, 100, 30, 10, 1], ["1000", "300", "100", "30", "10", "1"])
    ax.set_ylabel("hPa")
axes[-1].set_xlim(*span)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%Y"))
fig.colorbar(cf, ax=axes[1:], label="σ", shrink=0.8, pad=0.01)
save(fig, "winter", f"{w0}{YEAR}", "u60n10_pch")
plt.show()

### 6. The 2026/27 and 2027/28 winters

One row per start: every member's daily U(60°N, 10 hPa) (thin), the ensemble mean (thick), the
model's lead-time climatology (dashed) and ERA5 (climatology band and observations). Dots mark the
SSW central dates on the raw model wind.

In [ ]:
plume_span = (pd.Timestamp(YEAR, 8, 1), pd.Timestamp(YEAR + 2, 5, 31))
fig, axes = plt.subplots(len(INIT_MONTHS), 1, figsize=(16, 3.6 * len(INIT_MONTHS)), sharex=True, sharey=True,
                         gridspec_kw={"hspace": 0.25})
days = pd.date_range(*plume_span, freq="D")
dummy = xr.DataArray(np.zeros(days.size), dims="time", coords={"time": days.values})
cl, sd = on_days(clim_u60, dummy), on_days(std_u60, dummy)
model_ssw = {}                        # init -> DataFrame (series, winter, central_date) on the raw wind
model_ssw_adj = {}                    # bias-adjusted
for ax, init in zip(np.atleast_1d(axes), INIT_MONTHS):
    u = m_u60[init]
    ax.fill_between(days, cl - sd, cl + sd, color="0.82", label="ERA5 ±1σ")
    ax.plot(days, cl, color="0.35", lw=1.5, label="ERA5 climatology")
    for mem in u["member"].values:
        ax.plot(u["time"], u.sel(member=mem), color=INIT_COLORS[init], lw=0.5, alpha=0.35)
    ax.plot(u["time"], u.mean("member"), color=INIT_COLORS[init], lw=2.5, label="ensemble mean")
    ax.plot(u["time"], m_u60_clim[init], "--", color="k", lw=1.5, label="E3SM lead-time climatology")
    e_part = e_u60.sel(time=slice(*plume_span))
    if e_part.sizes["time"]:
        ax.plot(e_part["time"], e_part, "k-", lw=2.5, label="ERA5")
    tabs, tabs_adj = [], []
    for mem in u["member"].values:
        tabs.append(sc.ssw_table(u.sel(member=mem).values, u["time"].values, str(mem), **SSW))
        tabs_adj.append(sc.ssw_table(m_u60_adj[init].sel(member=mem).values, u["time"].values, str(mem), **SSW))
    model_ssw[init] = pd.concat(tabs, ignore_index=True)
    model_ssw_adj[init] = pd.concat(tabs_adj, ignore_index=True)
    ev = model_ssw[init]
    if len(ev):
        y = [float(u.sel(member=s, time=np.datetime64(d)).values) for s, d in zip(ev["series"], ev["central_date"])]
        ax.plot(ev["central_date"], np.asarray(y), "o", ms=6, mfc="w", mec="k", label="SSW (raw wind)")
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylim(-35, 70)
    ax.set_ylabel("m s$^{-1}$")
    ax.grid(alpha=0.3)
    ax.set_title(f"E3SM 1 {MONTH_NAMES[init]} {YEAR} start ({u.sizes['member']} members): U(60°N, {REF_LEVEL:g} hPa)", loc="left")
np.atleast_1d(axes)[0].legend(loc="upper left", ncol=3, fontsize=fontz * 0.65, frameon=False)
np.atleast_1d(axes)[-1].set_xlim(*plume_span)
np.atleast_1d(axes)[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
save(fig, "plume", "u60n10", YEAR)
plt.show()

**SSW frequency.** Events per member-winter for every fully covered winter (1 Nov - 30 Apr) of each
start, on the raw and the bias-adjusted wind, against ERA5 1979/80 to the last complete winter (all winters, and El Niño /
La Niña winters by DJF Niño3.4 from `NINO_PRODUCT_OBS`). The 90% intervals come from a bootstrap over winters (member-winters for the model).

In [ ]:
rng = np.random.default_rng(0)


def freq_row(label, counts, kind):
    counts = np.asarray(counts, dtype=float)
    boots = rng.choice(counts, size=(5000, counts.size)).mean(1) if counts.size else np.array([np.nan])
    return {"set": label, "wind": kind, "winters": counts.size, "events": int(counts.sum()),
            "events per winter": counts.mean() if counts.size else np.nan,
            "5%": np.percentile(boots, 5), "95%": np.percentile(boots, 95),
            "P(≥1 SSW)": (counts > 0).mean() if counts.size else np.nan}


def mid_month(times):
    # the 15th of each time's month (datetime64[ns])
    return sc.to_datetime64(times).astype("datetime64[M]").astype("datetime64[ns]") + np.timedelta64(14, "D")



# observed
obs_events = sc.ssw_table(e_u60.values, e_u60["time"].values, "ERA5", **SSW)
obs_winters = sc.covered_winters(e_u60["time"].values)
obs_counts = pd.Series(0, index=obs_winters).add(obs_events.groupby("winter").size(), fill_value=0).astype(int)
nino_obs = rc.box_mean(rc.obs_anomaly("TS", years=(ERA5_FIRST_YEAR, YEAR), mode=derivation_mode, product=NINO_PRODUCT_OBS),
                       rc.NINO_REGIONS["Nino3.4"]).load()
nino_obs = nino_obs.assign_coords(time=mid_month(nino_obs["time"].values))


def winter_mean(series, w, months=DJF):
    times = [np.datetime64(f"{w + dy}-{m:02d}-15") for dy, m in months]
    try:
        v = series.sel(time=times)
    except KeyError:
        return np.nan
    return float(v.mean()) if np.isfinite(v).all() else np.nan


obs_djf_nino = pd.Series({w: winter_mean(nino_obs, w) for w in obs_winters})
rows = [freq_row(f"ERA5 {obs_winters[0]}/{str(obs_winters[0] + 1)[2:]}-{obs_winters[-1]}/{str(obs_winters[-1] + 1)[2:]}",
                 obs_counts.values, "ERA5"),
        freq_row("ERA5 El Niño winters", obs_counts[obs_djf_nino >= ENSO_THRESHOLD].values, "ERA5"),
        freq_row("ERA5 La Niña winters", obs_counts[obs_djf_nino <= -ENSO_THRESHOLD].values, "ERA5")]
print("ERA5 El Niño winters:", ", ".join(f"{w}/{str(w + 1)[2:]}" for w in obs_counts[obs_djf_nino >= ENSO_THRESHOLD].index))

# model: per start and winter, raw and bias-adjusted
model_counts = {}                      # (init, winter, kind) -> per-member counts
for init in INIT_MONTHS:
    winters = sc.covered_winters(m_u60[init]["time"].values)
    members = [str(m) for m in m_u60[init]["member"].values]
    for kind, tab in (("raw", model_ssw[init]), ("bias-adjusted", model_ssw_adj[init])):
        for w in winters:
            c = tab[tab["winter"] == w].groupby("series").size().reindex(members, fill_value=0)
            model_counts[(init, w, kind)] = c
            rows.append(freq_row(f"E3SM {MONTH_NAMES[init]} start, {w}/{str(w + 1)[2:]}", c.values, kind))
for kind in ("raw", "bias-adjusted"):
    for w in (YEAR, YEAR + 1):
        pooled = [c.values for (i, ww, k), c in model_counts.items() if ww == w and k == kind]
        if pooled:
            rows.append(freq_row(f"E3SM all starts, {w}/{str(w + 1)[2:]}", np.concatenate(pooled), kind))
freq = pd.DataFrame(rows)
freq.to_csv(DATA_DIR / f"ssw_frequency_{YEAR}.csv", index=False)
print(f"[WRITE] {DATA_DIR / f'ssw_frequency_{YEAR}.csv'}")
display(freq.round(2))

events_all = pd.concat([obs_events.assign(source="ERA5")] +
                       [model_ssw[i].assign(source=f"E3SM init{i:02d}", wind="raw") for i in INIT_MONTHS] +
                       [model_ssw_adj[i].assign(source=f"E3SM init{i:02d}", wind="bias-adjusted") for i in INIT_MONTHS],
                       ignore_index=True)
events_all.to_csv(DATA_DIR / f"ssw_central_dates_{YEAR}.csv", index=False)
print(f"[WRITE] {DATA_DIR / f'ssw_central_dates_{YEAR}.csv'}")

In [ ]:
# SSW frequency, central-date months and final-warming dates
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), gridspec_kw={"width_ratios": [1.3, 1, 1.1], "wspace": 0.3})
ax = axes[0]
sel = freq[freq["set"].str.contains("ERA5|all starts")]
labels = [f"{s}\n({k})" if k != "ERA5" else s for s, k in zip(sel["set"], sel["wind"])]
colors = ["k" if k == "ERA5" else ("tab:red" if k == "raw" else "tab:blue") for k in sel["wind"]]
y = np.arange(len(sel))[::-1]
ax.errorbar(sel["events per winter"], y, xerr=[sel["events per winter"] - sel["5%"], sel["95%"] - sel["events per winter"]],
            fmt="none", ecolor=colors, capsize=4)
ax.scatter(sel["events per winter"], y, c=colors, s=60, zorder=3)
ax.set_yticks(y, labels, fontsize=fontz * 0.7)
ax.axvline(float(freq.iloc[0]["events per winter"]), color="k", ls=":", lw=1)
ax.set_xlabel("SSWs per winter (90% bootstrap)")
ax.set_title("(a) SSW frequency", loc="left")
ax.grid(alpha=0.3, axis="x")

ax = axes[1]
order = [11, 12, 1, 2, 3]
x = np.arange(len(order))
obs_m = obs_events["central_date"].dt.month.value_counts().reindex(order, fill_value=0) / len(obs_winters)
mod = pd.concat([model_ssw[i] for i in INIT_MONTHS])
n_mw = sum(len(c) for (i, w, k), c in model_counts.items() if k == "raw")
mod_m = (mod["central_date"].dt.month.value_counts().reindex(order, fill_value=0) / max(n_mw, 1)) if len(mod) else 0 * obs_m
ax.bar(x - 0.2, obs_m.values, 0.4, color="k", label=f"ERA5 ({len(obs_winters)} winters)")
ax.bar(x + 0.2, np.asarray(mod_m), 0.4, color="tab:red", label=f"E3SM raw ({n_mw} member-winters)")
ax.set_xticks(x, [MONTH_NAMES[m] for m in order])
ax.set_ylabel("SSWs per winter")
ax.set_title("(b) Month of the central date", loc="left")
ax.legend(frameon=False)
ax.grid(alpha=0.3, axis="y")

ax = axes[2]


def doy_spring(d):
    return (pd.Timestamp(d) - pd.Timestamp(pd.Timestamp(d).year, 1, 1)).days + 1


obs_fw_doy = [doy_spring(d) for d in obs_fw.values() if pd.notna(d)]
data, labels, cols = [obs_fw_doy], [f"ERA5\n{ERA5_FIRST_YEAR}-{obs_end.year}"], ["0.6"]
fw_rows = [{"source": "ERA5", "spring": y, "final_warming": d} for y, d in obs_fw.items()]
for init in INIT_MONTHS:
    u = m_u60[init]
    for spring in (YEAR, YEAR + 1, YEAR + 2):
        dates = [sc.final_warming_date(u.sel(member=m).values, u["time"].values, spring, recover_days=SSW["recover_days"])
                 for m in u["member"].values]
        if pd.Timestamp(spring, 1, 1) < pd.Timestamp(u["time"].values[0]) and spring == YEAR and init != 2:
            continue
        ok = [d for d in dates if pd.notna(d)]
        fw_rows += [{"source": f"E3SM init{init:02d}", "member": str(m), "spring": spring, "final_warming": d}
                    for m, d in zip(u["member"].values, dates)]
        if len(ok) >= 3:
            data.append([doy_spring(d) for d in ok])
            labels.append(f"{MONTH_NAMES[init]} start\n{spring}")
            cols.append(INIT_COLORS[init])
bp = ax.boxplot(data, vert=False, patch_artist=True, widths=0.6, whis=(5, 95))
for patch, c in zip(bp["boxes"], cols):
    patch.set_facecolor(c)
    patch.set_alpha(0.5)
if pd.notna(obs_fw.get(YEAR, pd.NaT)):
    ax.axvline(doy_spring(obs_fw[YEAR]), color="k", ls="--", lw=1.5, label=f"ERA5 {YEAR}: {obs_fw[YEAR]:%d %b}")
    ax.legend(frameon=False, loc="lower right")
ax.set_yticks(np.arange(1, len(labels) + 1), labels, fontsize=fontz * 0.7)
ticks = [pd.Timestamp(2001, m, 1) for m in (2, 3, 4, 5, 6)]
ax.set_xticks([doy_spring(t) for t in ticks], [f"1 {t:%b}" for t in ticks])
ax.set_title("(c) Final-warming date", loc="left")
ax.grid(alpha=0.3, axis="x")
save(fig, "ssw", "frequency", "final_warming", YEAR)
plt.show()

fw_table = pd.DataFrame(fw_rows)
fw_table.to_csv(DATA_DIR / f"final_warming_dates_{YEAR}.csv", index=False)
print(f"[WRITE] {DATA_DIR / f'final_warming_dates_{YEAR}.csv'}")

**DJF vortex strength.** Drift-corrected DJF-mean U(60°N, 10 hPa) anomalies of every member
(monthly output, the same as the drift climatology), with tercile probabilities. The tercile bounds
come from the model's own DJF anomalies in the Reanalysis hindcasts (Nov starts at leads 1-3, May
starts at leads 7-9; all years and members). ERA5 DJF anomalies 1979/80 onward are shown for
reference, with El Niño winters marked.

In [ ]:
# model terciles from the Reanalysis DJF anomalies
ref_djf = []
for m, leads in ((11, [1, 2, 3]), (5, [7, 8, 9])):
    if m in ref:
        u = sc.lat_mean(ref[m]["U"].sel(plev=REF_LEVEL), sc.U60).isel(lead=leads).mean("lead")   # (year, member)
        ref_djf.append((u - u.mean()).values.ravel())
ref_djf = np.concatenate(ref_djf)
ref_djf = ref_djf[np.isfinite(ref_djf)]
lo, hi = np.percentile(ref_djf, [100 / 3, 200 / 3])

# observed DJF anomalies and terciles
e_anom_month = e_u60_month - e_u60_month.sel(time=slice(f"{CLIM_YEARS[0]}-01", f"{CLIM_YEARS[1]}-12")).groupby("time.month").mean()
e_djf = pd.Series({w: winter_mean(e_anom_month.drop_vars("month"), w) for w in range(ERA5_FIRST_YEAR, YEAR)}).dropna()
e_lo, e_hi = np.percentile(e_djf.values, [100 / 3, 200 / 3])

m_djf, prob_rows = {}, []
for init in INIT_MONTHS:
    a = mm_u60_anom[init].assign_coords(time=monthly[init]["time"]).swap_dims(lead="time")
    for w in (YEAR, YEAR + 1):
        times = [np.datetime64(f"{w + dy}-{m:02d}-15") for dy, m in DJF]
        if not all(t in a["time"].values for t in times):
            continue
        v = a.sel(time=times).mean("time")
        m_djf[(init, w)] = v
        prob_rows.append({"start": MONTH_NAMES[init], "winter": f"{w}/{str(w + 1)[2:]}", "members": int(v.size),
                          "mean anomaly (m/s)": float(v.mean()), "P(weak)": float((v < lo).mean()),
                          "P(normal)": float(((v >= lo) & (v <= hi)).mean()), "P(strong)": float((v > hi).mean())})
probs = pd.DataFrame(prob_rows)
probs.to_csv(DATA_DIR / f"djf_vortex_terciles_{YEAR}.csv", index=False)
print(f"Model tercile bounds {lo:+.1f} / {hi:+.1f} m/s (Reanalysis, {ref_djf.size} member-winters); "
      f"ERA5 {e_lo:+.1f} / {e_hi:+.1f} m/s")
nino_years = [w for w in e_djf.index if obs_djf_nino.get(w, np.nan) >= ENSO_THRESHOLD]
print(f"ERA5 El Niño winters: weak {np.mean(e_djf[nino_years] < e_lo):.0%}, strong {np.mean(e_djf[nino_years] > e_hi):.0%} "
      f"({len(nino_years)} winters)")
display(probs.round(2))

fig, ax = plt.subplots(figsize=(13, 5))
pos, labels = [], []
for k, ((init, w), v) in enumerate(m_djf.items()):
    xs = np.full(v.size, k) + rng.uniform(-0.15, 0.15, v.size)
    ax.scatter(xs, v.values, color=INIT_COLORS[init], s=18, alpha=0.7)
    ax.plot([k - 0.3, k + 0.3], [float(v.mean())] * 2, color=INIT_COLORS[init], lw=3)
    pos.append(k)
    labels.append(f"{MONTH_NAMES[init]} start\n{w}/{str(w + 1)[2:]}")
k0 = len(pos) + 0.5
xs = np.full(e_djf.size, k0) + rng.uniform(-0.15, 0.15, e_djf.size)
nino_mask = np.array([w in nino_years for w in e_djf.index])
ax.scatter(xs[~nino_mask], e_djf.values[~nino_mask], color="0.5", s=18, label="ERA5 winters")
ax.scatter(xs[nino_mask], e_djf.values[nino_mask], color="tab:orange", edgecolor="k", s=40, label="ERA5 El Niño winters")
pos.append(k0)
labels.append(f"ERA5\n{int(e_djf.index[0])}-{int(e_djf.index[-1]) + 1}")
ax.axhspan(lo, hi, color="0.9", zorder=0, label="E3SM middle tercile")
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(pos, labels, fontsize=fontz * 0.75)
ax.set_ylabel("DJF U(60°N, 10 hPa) anomaly (m s$^{-1}$)")
ax.set_title("DJF polar-vortex strength: members (drift-corrected) and observed winters", loc="left")
ax.legend(frameon=False, loc="lower left", fontsize=fontz * 0.7)
ax.grid(alpha=0.3, axis="y")
save(fig, "djf", "vortex", "terciles", YEAR)
plt.show()

### 7. ENSO and QBO across members

Each member's DJF U(60°N, 10 hPa) anomaly against its own DJF Niño3.4 anomaly (drift-corrected
`TS`) and its Oct-Nov `QBO_LEVEL` equatorial wind, for the first forecast winter (2026/27). Lines are
least-squares fits over the pooled members (r and the slope in the legend). Grey points: ERA5
winters (HadISST 1.1 Niño3.4, ERA5 QBO). If the spread of the members is mostly internal, these
slopes are weak, and they test whether the model has the observed ENSO and Holton-Tan pathways.

In [ ]:
w = YEAR
nino_m, qbo_m = {}, {}
for init in INIT_MONTHS:
    if (init, w) not in m_djf:
        continue
    labels_m = [str(m) for m in m_djf[(init, w)]["member"].values]
    vals = {}
    for name, prefix in POOL.items():
        anom = rc.hindcast_anomaly(prefix, init, "TS", year=YEAR, method=CLIM_METHOD, mode=derivation_mode)
        if anom is None:
            continue
        n34 = rc.box_mean(anom, rc.NINO_REGIONS["Nino3.4"]).load()
        n34 = n34.assign_coords(time=("lead", mid_month(n34["time"].values))).swap_dims(lead="time")
        times = [np.datetime64(f"{w + dy}-{m:02d}-15") for dy, m in DJF]
        for mem in n34["member"].values:
            vals[f"{name}:{mem}"] = float(n34.sel(member=mem, time=times).mean())
    nino_m[init] = pd.Series({k: vals.get(k, np.nan) for k in labels_m})
    q = mm_qbo[init].sel(plev=QBO_LEVEL).assign_coords(time=monthly[init]["time"]).swap_dims(lead="time")
    qt = [np.datetime64(f"{w}-{m:02d}-15") for m in QBO_MONTHS]
    qbo_m[init] = q.sel(time=qt).mean("time").to_series().reindex(labels_m) if all(t in q["time"].values for t in qt) \
        else pd.Series(np.nan, index=labels_m)

e_qbo_on_s = pd.Series(e_qbo_on)
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
reg_rows = []
for ax, (xname, xm, xo, xl) in zip(axes, [("Niño3.4", nino_m, obs_djf_nino, "DJF Niño3.4 anomaly (°C)"),
                                          (f"QBO U{QBO_LEVEL:g}", qbo_m, e_qbo_on_s, f"Oct-Nov U {QBO_LEVEL:g} hPa, 5°S-5°N (m s$^{{-1}}$)")]):
    common = [y for y in e_djf.index if y in xo.index and np.isfinite(xo[y])]
    ax.scatter(xo[common], e_djf[common], color="0.6", s=25, label="ERA5 winters")
    if len(common) > 3:
        r = np.corrcoef(xo[common], e_djf[common])[0, 1]
        b = np.polyfit(xo[common], e_djf[common], 1)
        xx = np.linspace(xo[common].min(), xo[common].max(), 10)
        ax.plot(xx, np.polyval(b, xx), color="0.4", lw=2, ls="--", label=f"ERA5 r={r:+.2f}, slope {b[0]:+.1f}")
        reg_rows.append({"predictor": xname, "set": "ERA5", "n": len(common), "r": r, "slope": b[0]})
    allx, ally = [], []
    for init in xm:
        v = m_djf[(init, w)].to_series()
        v.index = [str(i) for i in v.index]
        x = xm[init].reindex(v.index)
        ok = np.isfinite(x.values) & np.isfinite(v.values)
        ax.scatter(x.values[ok], v.values[ok], color=INIT_COLORS[init], s=35, edgecolor="k", lw=0.3,
                   label=f"E3SM {MONTH_NAMES[init]} start")
        allx += list(x.values[ok])
        ally += list(v.values[ok])
    if len(allx) > 3:
        allx, ally = np.asarray(allx), np.asarray(ally)
        r = np.corrcoef(allx, ally)[0, 1]
        b = np.polyfit(allx, ally, 1)
        xx = np.linspace(allx.min(), allx.max(), 10)
        ax.plot(xx, np.polyval(b, xx), "k-", lw=2, label=f"E3SM pooled r={r:+.2f}, slope {b[0]:+.1f}")
        reg_rows.append({"predictor": xname, "set": f"E3SM {w}/{str(w + 1)[2:]} pooled", "n": allx.size, "r": r, "slope": b[0]})
    ax.axhline(0, color="k", lw=0.6)
    ax.axvline(0, color="k", lw=0.6)
    ax.set_xlabel(xl)
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=fontz * 0.65, loc="lower left")
axes[0].set_ylabel("DJF U(60°N, 10 hPa) anomaly (m s$^{-1}$)")
axes[0].set_title(f"(a) Vortex vs ENSO, winter {w}/{str(w + 1)[2:]}", loc="left")
axes[1].set_title(f"(b) Vortex vs QBO (Holton-Tan), winter {w}/{str(w + 1)[2:]}", loc="left")
save(fig, "drivers", "enso", "qbo", YEAR)
plt.show()
display(pd.DataFrame(reg_rows).round(2))

### 8. SSW composites

Lag composites around the central dates of all model SSWs (raw wind, every start and member-winter)
against ERA5 1979-2025. Panels: (a) U(60°N, 10 hPa); (b) 100-hPa eddy heat flux [v*T*], 45-75°N
(model: daily means, so sub-daily eddies are missing); (c, d) standardized polar-cap height.
The time-height panels show how far down the anomaly reaches and how long it lasts.

In [ ]:
e_comp = {"u": sc.lag_composite(e_u60, obs_events["central_date"], LAGS),
          "vt": sc.lag_composite(e_vt, obs_events["central_date"], LAGS),
          "pch": sc.lag_composite(e_pch, obs_events["central_date"], LAGS)}
m_parts = {"u": [], "vt": [], "pch": []}
for init in INIT_MONTHS:
    for _, ev in model_ssw[init].iterrows():
        mem = ev["series"]
        for key, src in (("u", m_u60[init]), ("vt", m_vt[init]), ("pch", m_pch[init])):
            m_parts[key].append(sc.lag_composite(src.sel(member=mem), [ev["central_date"]], LAGS))
m_comp = {k: xr.concat(v, dim="event") if v else None for k, v in m_parts.items()}
n_obs, n_mod = len(obs_events), (m_comp["u"].sizes["event"] if m_comp["u"] is not None else 0)
print(f"composites: ERA5 {n_obs} SSWs, E3SM {n_mod} SSWs")

fig = plt.figure(figsize=(17, 10))
gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.18)
for k, (key, title, units) in enumerate([("u", f"(a) U(60°N, {REF_LEVEL:g} hPa)", "m s$^{-1}$"),
                                         ("vt", "(b) [v*T*] 100 hPa, 45-75°N", "K m s$^{-1}$")]):
    ax = fig.add_subplot(gs[0, k])
    for comp, color, label, n in ((e_comp[key], "k", "ERA5", n_obs), (m_comp[key], "tab:red", "E3SM", n_mod)):
        if comp is None:
            continue
        mu, q = comp.mean("event"), comp.quantile([0.1, 0.9], "event")
        ax.fill_between(LAGS, q.isel(quantile=0), q.isel(quantile=1), color=color, alpha=0.15)
        ax.plot(LAGS, mu, color=color, lw=2.5, label=f"{label} ({n} events; 10-90%)")
    ax.axvline(0, color="0.4", lw=1)
    ax.axhline(0, color="0.4", lw=0.8)
    ax.set_xlabel("days from central date")
    ax.set_ylabel(units)
    ax.set_title(title, loc="left")
    ax.legend(frameon=False, fontsize=fontz * 0.75)
    ax.grid(alpha=0.3)
pch_lev = np.array([-2.5, -2, -1.5, -1, -0.5, 0.5, 1, 1.5, 2, 2.5])
lag_sel = slice(*PCH_LAGS)
for k, (comp, label, n) in enumerate(((e_comp["pch"], "ERA5", n_obs), (m_comp["pch"], "E3SM", n_mod))):
    ax = fig.add_subplot(gs[1, k])
    if comp is None:
        ax.axis("off")
        continue
    c = comp.mean("event").sel(lag=lag_sel).sel(plev=slice(1, 1000)).dropna("plev", how="all")
    cf = ax.contourf(c["lag"], c["plev"], c.T, levels=pch_lev, cmap="RdBu_r", extend="both")
    ax.set_yscale("log")
    ax.set_ylim(1000, 1 if label == "ERA5" else 10)
    ax.set_yticks([1000, 300, 100, 30, 10, 3, 1], ["1000", "300", "100", "30", "10", "3", "1"])
    ax.axvline(0, color="k", lw=1)
    ax.set_xlabel("days from central date")
    ax.set_ylabel("hPa")
    ax.set_title(f"({'cd'[k]}) {label} polar-cap height anomaly / σ ({n} events)", loc="left")
fig.colorbar(cf, ax=fig.axes[2:], label="σ (ERA5 daily)", shrink=0.8, pad=0.01)
save(fig, "ssw", "composite", YEAR)
plt.show()

# downward coupling: mean standardized 1000-hPa polar-cap height over days +5..+60 after the central date
dc = {lab: float(c.sel(plev=1000.0, lag=slice(5, 60)).mean()) for lab, c in (("ERA5", e_comp["pch"]), ("E3SM", m_comp["pch"]))
      if c is not None}
print("Mean 1000-hPa polar-cap height anomaly, days +5..+60 (σ):", {k: round(v, 2) for k, v in dc.items()})